# Demo 2: Prefect – Python-first orchestration

**Kapcsolódó diák:** 25–27 (Prefect architektúra, flows, tasks, deployments)

**Témák ebben a notebookban:**
1. Airflow vs Prefect szemléletmód – összehasonlítás
2. Az első `@flow` – legegyszerűbb 5 soros példa
3. `@task` paraméterek – `retries`, `log_prints`, `name`
4. Return value = automatikus adatátadás (Airflow XCom vs Prefect return)
5. Task caching – `task_input_hash` fogalom
6. Retry `@task`-ban – exponential backoff szimulációval
7. Subflow – mikor érdemes kompozíciót használni?
8. Deployment alapok – `prefect deploy` parancsok
9. Prefect Cloud vs self-hosted összehasonlítás
10. Összefoglalás – mikor válasszuk a Prefect-et?

In [1]:
!pip install prefect==2.19.0

  Using cached anyio-3.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached importlib_resources-6.1.3-py3-none-any.whl.metadata (3.9 kB)
  Using cached uvicorn-0.28.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached pendulum-2.1.2-cp311-cp311-manylinux_2_35_x86_64.whl
  Using cached pytz-2024.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
INFO: pip is looking at multiple versions of httpx to determine which version is compatible with other requirements. This could take a while.
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
INFO: pip is looking at multiple versions of httpx[http2] to determine which version is compatible with other requirements. This could take a while.
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached anyio-3.7.1-py3-none-any.whl (80 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using c

## 1. Airflow vs Prefect szemléletmód

| Szempont | Airflow | Prefect |
|---|---|---|
| DAG definíció | Python fájl, statikus gráf `>>` operátorral | Python függvény, dinamikus gráf futáskor épül |
| Scheduler | Külön service (mindig fut) | Nincs szükség rá lokális futtatáshoz |
| Paraméterátadás | XCom push/pull (explicit) | Return value (implicit, natív Python) |
| Telepítés | docker-compose, 6+ service | `pip install prefect` + 1 parancs |
| UI | Airflow Webserver | Prefect UI / Prefect Cloud |
| Tanulási görbe | Meredek (DAG szemlélet) | Lapos (Python szemlélet) |

**Kulcskülönbség:** Airflow-ban a DAG struktúráját a fájl betöltésekor kell meghatározni
(statikus). Prefect-ben a flow futáskor, dinamikusan épül fel – például ciklusok,
feltételek, runtime paraméterek alapján.

In [2]:
# Airflow vs Prefect kód-összehasonlítás – csak string demo, nincs import

airflow_code = '''
# ── AIRFLOW ──────────────────────────────────────
from airflow.decorators import dag, task
from datetime import datetime

@dag(schedule='@daily', start_date=datetime(2024,1,1))
def etl_dag():
    @task
    def extract(): return [1, 2, 3]

    @task
    def load(data): print(data)       # XCom implicit itt

    load(extract())                    # gráf statikus, betöltéskor rögzül

etl_dag()  # DAG objektum regisztrálása – nem fut le!
'''

prefect_code = '''
# ── PREFECT ──────────────────────────────────────
from prefect import flow, task

@task
def extract(): return [1, 2, 3]

@task
def load(data): print(data)        # return érték átadása natívan

@flow
def etl_flow():
    data = extract()                   # gráf dinamikus, futáskor épül
    load(data)

etl_flow()  # azonnal lefut – Scheduler nem kell!
'''

print('AIRFLOW kód:')
print(airflow_code)
print('PREFECT kód:')
print(prefect_code)

AIRFLOW kód:

# ── AIRFLOW ──────────────────────────────────────
from airflow.decorators import dag, task
from datetime import datetime

@dag(schedule='@daily', start_date=datetime(2024,1,1))
def etl_dag():
    @task
    def extract(): return [1, 2, 3]

    @task
    def load(data): print(data)       # XCom implicit itt

    load(extract())                    # gráf statikus, betöltéskor rögzül

etl_dag()  # DAG objektum regisztrálása – nem fut le!

PREFECT kód:

# ── PREFECT ──────────────────────────────────────
from prefect import flow, task

@task
def extract(): return [1, 2, 3]

@task
def load(data): print(data)        # return érték átadása natívan

@flow
def etl_flow():
    data = extract()                   # gráf dinamikus, futáskor épül
    load(data)

etl_flow()  # azonnal lefut – Scheduler nem kell!



### Mi a különbség?

- **Airflow:** a `@dag` dekorátor egy DAG objektumot regisztrál a Scheduler számára.
  A Python kód nem fut le azonnal – csak a gráf struktúráját írja le.
  A Scheduler ütemezi és a Worker futtatja.

- **Prefect:** a `@flow` dekorátor egy szokásos Python függvényt csomagol be (*wrapper*).
  Meghíváskor **azonnal lefut** – pontosan úgy, mint egy normál Python hívás.
  A Prefect a háttérben naplóz, figyeli az állapotot, kezeli a hibákat.

- **Dinamikus gráf:** Prefect-ben a task-ok végrehajtási sorrendje a `return` értékek
  függőségeiből következik – nincs szükség explicit `>>` operátorra.

## 2. Az első `@flow` – legegyszerűbb 5 soros példa

A Prefect minimális használatához csak két dekorátor kell: `@flow` és `@task`.
Nincs konfigurációs fájl, nincs adatbázis, nincs Scheduler – csak Python.

In [3]:
# Bare minimum @flow + @task – a lehető legegyszerűbb Prefect példa
try:
    from prefect import flow, task

    @task                              # @task: újrafelhasználható munkaegység
    def greet(name: str) -> str:
        return f'Szia, {name}!'

    @flow                              # @flow: orchestration egység
    def hello_flow(name: str = 'Világ'):
        msg = greet(name)              # task hívás a flow-n belül
        print(msg)                     # print: megjelenik a logban

    hello_flow(name='Adatmérnök')      # közvetlen meghívás – azonnal lefut
    hello_flow()                       # default paraméterrel

except ImportError:
    print('Telepítés szükséges: pip install prefect')

Telepítés szükséges: pip install prefect


### Mi történt? – soronkénti magyarázat

- `@task`: a `greet` függvény köré Prefect wrapper kerül.
  A wrapper gondoskodik a naplózásról, cache-ről, retry-ról.
- `@flow`: a `hello_flow` szintén wrapper – de ez az orchestration egység.
  Figyeli az összes belső task állapotát, kezeli a hibákat.
- `msg = greet(name)`: a task hívása a flow-n belül **nem** egy sima függvényhívás –
  a Prefect nyomkövet minden hívást (state: `Running → Completed`).
- `hello_flow(name='Adatmérnök')`: a flow meghívása azonnal lefut, visszaad egy
  `State` objektumot (de `print` esetén látjuk az outputot is).

## 3. `@task` paraméterek – `retries`, `log_prints`, `name`

A `@task` dekorátor számos paramétert fogad:

| Paraméter | Típus | Leírás |
|---|---|---|
| `name` | str | A task neve a UI-ban és a logban |
| `retries` | int | Hány újrakísérlet hiba esetén |
| `retry_delay_seconds` | int/float | Várakozás újrakísérletek között |
| `log_prints` | bool | `print()` automatikusan a Prefect logba kerül |
| `timeout_seconds` | int | Task futási időkorlát |
| `cache_key_fn` | callable | Cache kulcs számítása (ld. 5. fejezet) |

In [4]:
# @task paraméterek teljes demo
try:
    from prefect import flow, task
    import random

    attempt_counter = {'n': 0}         # egyszerű számláló a retry szemléltetéséhez

    @task(
        name='adat-feldolgozas',          # megjelenik a Prefect UI-ban
        retries=2,                         # 2 újrakísérlet hiba esetén
        retry_delay_seconds=1,             # 1 mp várakozás (demo: rövid)
        log_prints=True,                   # print() → Prefect log
    )
    def process_data(value: int) -> int:
        attempt_counter['n'] += 1
        print(f'  Kísérlet #{attempt_counter["n"]} – érték: {value}')
        if attempt_counter['n'] < 2:       # első kísérlet szándékosan hibás
            raise ValueError('Szimulált hiba az 1. kísérleten')
        result = value * 2
        print(f'  Eredmény: {result}')
        return result

    @flow(name='task-params-demo', log_prints=True)
    def demo_flow():
        print('Flow indul...')
        r = process_data(21)               # retry miatt 2x fut
        print(f'Flow végeredmény: {r}')

    demo_flow()

except ImportError:
    print('Telepítés szükséges: pip install prefect')

Telepítés szükséges: pip install prefect


### Mi változott? – Airflow vs Prefect retry összehasonlítás

| | Airflow | Prefect |
|---|---|---|
| Retry konfig helye | DAG szintű `default_args` vagy `@task(retries=...)` | `@task(retries=..., retry_delay_seconds=...)` |
| Retry láthatóság | Airflow UI → Task Instance nézet | Prefect UI → Flow Run → Task Run |
| Exponential backoff | `retry_exponential_backoff=True` | `retry_delay_seconds` + manuális logika |
| `log_prints` | Nincs, mindig stdout | Opcionális – `log_prints=True` |

**Prefect előny:** a retry konfiguráció közvetlenül a task mellett van a kódban –
nem kell globális `default_args`-t keresni.

## 4. Return value = automatikus adatátadás

Airflow-ban a task-ok közötti adatátadás **XCom** mechanizmuson keresztül történik:
```python
# Airflow XCom – explicit push/pull
ti.xcom_push(key='records', value=data)
data = ti.xcom_pull(task_ids='extract', key='records')
```

Prefect-ben ez nem kell – a task **return értéke** automatikusan átadódik:
```python
# Prefect – natív Python return
raw = extract_data(date)       # raw = a task visszatérési értéke
valid = validate_data(raw)     # raw átadva paraméterként
```

A Prefect a háttérben **Result** objektumban tárolja az értékeket,
de a fejlesztő ezt nem látja – pontosan úgy néz ki, mint normál Python kód.

In [5]:
# Teljes ETL flow: extract → validate → load, return értékek átadása
try:
    from prefect import flow, task
    import random, hashlib

    @task(log_prints=True, name='extract')
    def extract_data(date: str) -> list:
        """Szimulált API hívás – napi tranzakciók letöltése."""
        random.seed(int(hashlib.md5(date.encode()).hexdigest(), 16) % 9999)
        records = [{'id': i, 'amount': round(random.uniform(100, 50000), 2),
                    'date': date, 'valid': random.random() > 0.1}
                   for i in range(1, 21)]
        print(f'[extract] Letöltve: {len(records)} rekord ({date})')
        return records                     # ← return érték = következő task inputja

    @task(log_prints=True, name='validate')
    def validate_data(records: list) -> list:
        """Szűrés: csak érvényes, pozitív összegű sorok."""
        valid = [r for r in records if r['valid'] and r['amount'] > 0]
        print(f'[validate] {len(valid)} / {len(records)} rekord érvényes')
        return valid                       # ← return érték átadódik load_data-nak

    @task(log_prints=True, name='load')
    def load_data(records: list) -> dict:
        """Szimulált DB insert – összesítő statisztika."""
        total = sum(r['amount'] for r in records)
        print(f'[load] Betöltve: {len(records)} sor | Összeg: {total:,.0f} Ft')
        return {'rows': len(records), 'total': total}

    @flow(name='etl-pipeline', log_prints=True)
    def etl_pipeline(date: str = '2024-01-15') -> dict:
        raw    = extract_data(date)        # 1. task – return: raw list
        valid  = validate_data(raw)        # 2. task – raw átadva paraméterként
        result = load_data(valid)          # 3. task – valid átadva paraméterként
        return result

    # Közvetlen futtatás – Scheduler nélkül
    out = etl_pipeline(date='2024-01-15')
    print(f'\nFlow eredmény: {out}')

except ImportError:
    print('Telepítés szükséges: pip install prefect')

Telepítés szükséges: pip install prefect


### Mi történt? – Return value vs XCom push/pull

- `raw = extract_data(date)`: a task visszatér egy listával.
  Prefect a háttérben `State(result=raw_list)` objektumban tárolja.
- `valid = validate_data(raw)`: a `raw` lista **közvetlenül** átadható
  a következő task paraméterének – nincs `xcom_pull`.
- **Airflow XCom korlátja:** alapértelmezetten a metaadatbázisban tárolódik
  (max ~48 KB). Nagy adatoknál külön backend kell (S3, GCS).
- **Prefect Result:** alapértelmezetten in-memory, de konfigurálható
  `LocalFileSystemResultStorage` vagy `S3ResultStorage` használatára.

## 5. Task caching – `task_input_hash` fogalom

`cache_key_fn=task_input_hash` azt jelenti: **az input paraméterek hash-éből**
számítja a cache kulcsot. Ha ugyanazokkal a paraméterekkel hívják meg a task-ot
és a cache még érvényes → az előző eredményt adja vissza, **nem fut újra**.

```python
from prefect.tasks import task_input_hash
from datetime import timedelta

@task(
    cache_key_fn=task_input_hash,       # hash(input paraméterek)
    cache_expiration=timedelta(hours=1) # 1 óra után lejár a cache
)
def draga_muvelet(adat: str) -> str: ...
```

**Mikor hasznos?**
- Drága API hívások (pl. ML feature extraction, geocoding)
- Idempotens transzformációk: ha a forrás nem változott, az eredmény sem változott
- Fejlesztés/debug közben: gyorsabb iteráció, nem kell minden task újrafutni

In [6]:
# Cache demonstráció – task_input_hash: 2x azonos input → 2. gyorsabb
try:
    from prefect import flow, task
    from prefect.tasks import task_input_hash
    from datetime import timedelta
    import time, random, hashlib

    @task(
        cache_key_fn=task_input_hash,          # cache kulcs = hash(input)
        cache_expiration=timedelta(minutes=5), # 5 perc után lejár
        log_prints=True,
    )
    def slow_extract(date: str) -> list:
        """Szimulált lassú API – cache nélkül mindig újrafutna."""
        time.sleep(0.5)                        # szimulált hálózati késleltetés
        random.seed(int(hashlib.md5(date.encode()).hexdigest(), 16) % 9999)
        records = [{'id': i, 'v': round(random.random(), 4)} for i in range(10)]
        print(f'[slow_extract] LEFUTOTT ({date}) – {len(records)} rekord')
        return records

    @flow(log_prints=True)
    def cache_demo_flow():
        print('--- 1. hívás (cache miss – lefut) ---')
        t0 = time.time()
        r1 = slow_extract('2024-03-01')
        print(f'Idő: {time.time()-t0:.2f}s | Sorok: {len(r1)}')

        print('\n--- 2. hívás, azonos dátum (cache HIT – nem fut újra) ---')
        t1 = time.time()
        r2 = slow_extract('2024-03-01')         # cache hit!
        print(f'Idő: {time.time()-t1:.4f}s | Sorok: {len(r2)}')

        print('\n--- 3. hívás, más dátum (cache miss – lefut) ---')
        t2 = time.time()
        r3 = slow_extract('2024-03-02')         # más input → más hash → cache miss
        print(f'Idő: {time.time()-t2:.2f}s | Sorok: {len(r3)}')

    cache_demo_flow()

except ImportError:
    print('Telepítés szükséges: pip install prefect')

Telepítés szükséges: pip install prefect


### Mi történt? – `cache_key_fn` hash számítás + `cache_expiration`

- **1. hívás (`2024-03-01`):** nincs cache bejegyzés → task lefut (~0.5 s).
  A Prefect elmenti: `hash('2024-03-01') → [rekordok]`.
- **2. hívás (`2024-03-01`):** ugyanaz a hash → cache hit →
  az előző eredményt adja vissza **azonnal** (~0.001 s).
- **3. hívás (`2024-03-02`):** más input → más hash → cache miss → task lefut.
- `cache_expiration=timedelta(minutes=5)`: 5 perc után a cache bejegyzés
  érvénytelenné válik, a task újra lefut (akkor is, ha az input ugyanaz).

In [7]:
# Manuális cache key függvény – egyedi hash logika
# Pl.: csak a dátum első 7 karaktere számít (havi granularitás)
try:
    from prefect import flow, task
    import hashlib

    def monthly_cache_key(context, parameters):
        """Cache kulcs: csak az év-hónap számít, nem a pontos dátum."""
        date_str = parameters.get('date', '')
        month_key = date_str[:7]               # pl. '2024-03' (nap levágva)
        key = hashlib.md5(month_key.encode()).hexdigest()
        print(f'  [cache_key] date={date_str!r} → month={month_key!r} → key={key[:8]}...')
        return key

    @task(cache_key_fn=monthly_cache_key, log_prints=True)
    def monthly_report(date: str) -> str:
        """Havi riport – azonos hónapon belül cache-ből tér vissza."""
        print(f'  [monthly_report] LEFUTOTT – {date}')
        return f'Riport: {date[:7]}'

    @flow(log_prints=True)
    def manual_cache_flow():
        print('Március 1. (cache miss):')
        r1 = monthly_report('2024-03-01')
        print(f'  Eredmény: {r1}')
        print('Március 15. (azonos hónap → cache HIT):')
        r2 = monthly_report('2024-03-15')
        print(f'  Eredmény: {r2}')
        print('Április 1. (más hónap → cache miss):')
        r3 = monthly_report('2024-04-01')
        print(f'  Eredmény: {r3}')

    manual_cache_flow()

except ImportError:
    print('Telepítés szükséges: pip install prefect')

Telepítés szükséges: pip install prefect


## 6. Retry `@task`-ban – exponential backoff

A Prefect 2.x-ben a `retry_delay_seconds` lehet egyszerű szám vagy lista.
Lista esetén minden kísérlethez külön késleltetés adható meg – ez az **exponential backoff**:

```python
@task(
    retries=3,
    retry_delay_seconds=[1, 5, 30]   # 1s → 5s → 30s
)
def instabil_api_hivas(): ...
```

**Mikor szükséges?** Hálózati hibák, rate limiting, ideiglenes DB kapcsolat megszakadás esetén.
Az exponential backoff csökkenti a szerver terhelését: egyre ritkábban próbálkozik újra.

In [8]:
# Flaky task retry szimuláció: 50% eséllyel dob RuntimeError, retries=3
try:
    from prefect import flow, task
    import random

    call_log = []                          # hívások naplója

    @task(
        retries=3,                             # max 3 újrakísérlet
        retry_delay_seconds=1,                 # demo: 1 mp (élesben pl. [1, 5, 30])
        log_prints=True,
    )
    def flaky_api_call(endpoint: str) -> dict:
        """50% eséllyel hibázó API-hívás szimulációja."""
        attempt = len(call_log) + 1
        call_log.append(attempt)
        print(f'  API hívás #{attempt} – endpoint: {endpoint}')
        if random.random() < 0.5:             # 50% hiba valószínűség
            print(f'  ✗ Hiba a #{attempt}. kísérleten – újrapróbálás...')
            raise RuntimeError(f'Időtúllépés a {endpoint} végponton')
        print(f'  ✓ Sikeres a #{attempt}. kísérleten')
        return {'status': 'ok', 'attempt': attempt}

    @flow(log_prints=True)
    def retry_demo_flow():
        random.seed(42)                        # reprodukálható eredmény
        result = flaky_api_call('/api/data')
        print(f'\nVégeredmény: {result}')
        print(f'Összes kísérlet: {len(call_log)}')

    retry_demo_flow()

except ImportError:
    print('Telepítés szükséges: pip install prefect')

Telepítés szükséges: pip install prefect


## 7. Subflow – mikor érdemes kompozíciót használni?

A Prefect-ben egy `@flow` meghívhat más `@flow`-kat – ezek lesznek a **subflow-k**.
Ez lehetővé teszi a moduláris pipeline-tervezést:

- Minden üzleti folyamat **saját flow**-ban → önállóan tesztelhető, monitorozható
- A **master flow** koordinálja őket → áttekinthető top-level logika
- Subflow-k **újrafuttathatók** önállóan (ha pl. csak a load lépés hibázott)
- Subflow-k **más Work Pool-on** is futhatnak (pl. extract: Kubernetes, load: process)

In [9]:
# Master @flow 3 subflow-val: extract / transform / load külön flowként
try:
    from prefect import flow, task

    # ── Subflow 1: extract ──────────────────────────────────────────────────
    @flow(name='extract-subflow', log_prints=True)
    def extract_subflow(date: str) -> list:
        print(f'  [extract] {date} – 10 rekord letöltve')
        return [{'id': i, 'val': float(i * 100), 'date': date}
                for i in range(1, 11)]

    # ── Subflow 2: transform ────────────────────────────────────────────────
    @flow(name='transform-subflow', log_prints=True)
    def transform_subflow(records: list) -> list:
        converted = [{'id': r['id'], 'val_huf': r['val'] * 390, 'date': r['date']}
                     for r in records]
        print(f'  [transform] {len(converted)} rekord konvertálva (USD→HUF)')
        return converted

    # ── Subflow 3: load ─────────────────────────────────────────────────────
    @flow(name='load-subflow', log_prints=True)
    def load_subflow(records: list) -> int:
        total = sum(r['val_huf'] for r in records)
        print(f'  [load] {len(records)} sor | Összeg: {total:,.0f} HUF')
        return len(records)

    # ── Master flow: koordinálja a subflow-kat ──────────────────────────────
    @flow(name='master-etl', log_prints=True)
    def master_pipeline(dates: list) -> dict:
        """Master flow: több napot dolgoz fel szubflow-kkal."""
        summary = {}
        for date in dates:
            print(f'\nFeldolgozás: {date}')
            raw         = extract_subflow(date)        # subflow hívás 1
            transformed = transform_subflow(raw)       # subflow hívás 2
            n           = load_subflow(transformed)    # subflow hívás 3
            summary[date] = n
        return summary

    eredmeny = master_pipeline(['2024-01-01', '2024-01-02', '2024-01-03'])
    print(f'\nÖsszesítő: {eredmeny}')

except ImportError:
    print('Telepítés szükséges: pip install prefect')

Telepítés szükséges: pip install prefect


### Mi a subflow előnye? – monitorozás és újrafuttathatóság

- **Monitorozás:** a Prefect UI-ban minden subflow külön "flow run"-ként jelenik meg,
  saját log-gal, állapottal, futási idővel → könnyű megtalálni, melyik lépés hibázott.
- **Újrafuttathatóság:** ha `load_subflow` hibázott, csak azt kell újrafuttatni –
  nem kell az egész master flow-t újrakezdeni.
- **Kódmegosztás:** ugyanaz a `transform_subflow` több master flow-ból is meghívható.
- **Skálázhatóság:** subflow-k különböző Work Pool-okon futhatnak (pl. GPU-igényes
  ML step Kubernetes-en, egyszerű load step helyi process-en).

## 8. Deployment alapok – `prefect deploy` parancsok

A **deployment** = flow + ütemezés + futtatási környezet leírása.
Egy flow-ból több deployment is lehet (pl. napi prod + óránkénti staging).

```bash
# 1. Prefect szerver indítása (önhostolt, lokális)
prefect server start
# UI: http://localhost:4200

# 2. Work Pool létrehozása (process típusú – lokálisan futtat)
prefect work-pool create --type process default-pool

# 3. Worker indítása (ez veszi fel és futtatja a deploymenteket)
prefect worker start --pool default-pool

# 4. Deployment létrehozása CLI-ből
prefect deploy pipeline.py:etl_pipeline \\
    --name etl-daily \\
    --cron '0 3 * * *' \\
    --pool default-pool

# 5. Deployment azonnali manuális indítása
prefect deployment run etl-pipeline/etl-daily
```

In [10]:
# Deployment YAML konfiguráció kinyomtatása (nem kell Prefect server)

deployment_yaml = '''\
# prefect.yaml – deployment konfiguráció fájl
name: etl-daily-prod
version: '1.0'
entrypoint: pipelines/etl.py:etl_pipeline

# Ütemezés: minden nap 03:00-kor Budapest időzónában
schedules:
  - cron: "0 3 * * *"
    timezone: "Europe/Budapest"
    active: true

# Futtatási környezet
work_pool:
  name: default-pool
  work_queue_name: default
  job_variables:
    image: myrepo/etl-pipeline:latest  # Docker image (docker pool esetén)

# Alapértelmezett paraméterek
parameters:
  date: null          # runtime-ban adják meg, vagy yesterday()
  env: production

# Pull lépések: hogyan töltse le a kódot a worker
pull:
  - prefect.deployments.steps.git_clone:
      repository: https://github.com/myorg/pipelines.git
      branch: main
'''

print('=== prefect.yaml tartalma ===')
print(deployment_yaml)

# Work Pool típusok összehasonlítása
pool_types = [
    ('process',    'Lokálisan futtat OS process-ként',          'Fejlesztés, kis terhelés'),
    ('docker',     'Docker konténerben futtat',                  'Izolált env, CI/CD'),
    ('kubernetes', 'K8s Pod-ként futtat',                        'Prod, auto-scaling'),
    ('ecs',        'AWS ECS task-ként futtat',                   'AWS natív megoldás'),
]
print('\n=== Work Pool típusok ===')
print(f'{"Típus":<14} {"Leírás":<42} {"Mikor?"}')
print('-' * 75)
for t, desc, when in pool_types:
    print(f'{t:<14} {desc:<42} {when}')

=== prefect.yaml tartalma ===
# prefect.yaml – deployment konfiguráció fájl
name: etl-daily-prod
version: '1.0'
entrypoint: pipelines/etl.py:etl_pipeline

# Ütemezés: minden nap 03:00-kor Budapest időzónában
schedules:
  - cron: "0 3 * * *"
    timezone: "Europe/Budapest"
    active: true

# Futtatási környezet
work_pool:
  name: default-pool
  work_queue_name: default
  job_variables:
    image: myrepo/etl-pipeline:latest  # Docker image (docker pool esetén)

# Alapértelmezett paraméterek
parameters:
  date: null          # runtime-ban adják meg, vagy yesterday()
  env: production

# Pull lépések: hogyan töltse le a kódot a worker
pull:
  - prefect.deployments.steps.git_clone:
      repository: https://github.com/myorg/pipelines.git
      branch: main


=== Work Pool típusok ===
Típus          Leírás                                     Mikor?
---------------------------------------------------------------------------
process        Lokálisan futtat OS process-ként           Fejlesztés

### Work Pool típusok: `process` vs `docker` vs `kubernetes`

| Típus | Hogyan futtat | Előny | Hátrány |
|---|---|---|---|
| `process` | OS subprocess | Egyszerű, nincs overhead | Nincs izoláció, nincs skálázás |
| `docker` | Docker container | Izolált env, hordozható | Docker daemon kell |
| `kubernetes` | K8s Pod | Auto-scaling, prod-kész | K8s cluster kell |
| `ecs` | AWS ECS Task | AWS natív, serverless | AWS lock-in |

**Ajánlás:** fejlesztéshez `process`, staging/prod-hoz `docker` vagy `kubernetes`.
A deployment YAML-ban a `work_pool.name` meghatározza, melyik pool veszi fel a task-ot.

## 9. Prefect Cloud vs self-hosted összehasonlítás

| Szempont | Self-hosted (prefect server) | Prefect Cloud |
|---|---|---|
| Telepítés | `prefect server start` | Nincs – SaaS |
| Ár | Ingyenes | Ingyenes tier + fizetős |
| UI | Lokális (localhost:4200) | cloud.prefect.io |
| Adatok | Saját infrastruktúra | Prefect szerverein |
| HA / DR | Saját felelősség | Biztosított |
| Automations | Korlátozott | Teljes (webhooks, alerts) |
| SSO / RBAC | Nincs | Fizetős tervekben |

**Mikor melyiket?**
- **Self-hosted:** adatvédelmi követelmények, air-gapped env, teljes kontroll
- **Prefect Cloud:** gyors start, kis csapat, nincs infra kapacitás, Prefect Automations kell

In [11]:
# Prefect server CLI parancsok demonstrálása (print szimulációval)
# (Valós futtatáshoz: terminálban kell ezeket kiadni)

commands = [
    ('prefect server start',
     'Lokális Prefect szerver indítása (UI: http://localhost:4200)'),
    ('prefect work-pool create --type process default-pool',
     'Process típusú Work Pool létrehozása'),
    ('prefect worker start --pool default-pool',
     'Worker indítása – ez veszi fel és futtatja a deployment-eket'),
    ('prefect deploy pipeline.py:etl_pipeline --name etl-daily --cron "0 3 * * *"',
     'Deployment regisztrálása ütemezéssel'),
    ('prefect deployment run etl-pipeline/etl-daily',
     'Deployment azonnali manuális indítása'),
    ('prefect flow-run ls',
     'Legutóbbi flow futások listázása'),
    ('prefect deployment ls',
     'Összes deployment listázása'),
]

print('=== Prefect CLI parancsok ===')
for cmd, desc in commands:
    print(f'\n$ {cmd}')
    print(f'  # {desc}')

=== Prefect CLI parancsok ===

$ prefect server start
  # Lokális Prefect szerver indítása (UI: http://localhost:4200)

$ prefect work-pool create --type process default-pool
  # Process típusú Work Pool létrehozása

$ prefect worker start --pool default-pool
  # Worker indítása – ez veszi fel és futtatja a deployment-eket

$ prefect deploy pipeline.py:etl_pipeline --name etl-daily --cron "0 3 * * *"
  # Deployment regisztrálása ütemezéssel

$ prefect deployment run etl-pipeline/etl-daily
  # Deployment azonnali manuális indítása

$ prefect flow-run ls
  # Legutóbbi flow futások listázása

$ prefect deployment ls
  # Összes deployment listázása


## 10. Összefoglalás – mikor válasszuk a Prefect-et?

### Prefect erősségei
- **Python-natív:** nincs DAG-fájl, nincs operátor-könyvtár – csak Python
- **Dinamikus pipeline:** futáskor épül a gráf → ciklusok, feltételek természetesen
- **Gyors start:** `pip install prefect` + 5 sor kód = működő pipeline
- **Built-in cache, retry:** dekorátor paraméterekkel
- **Return value:** nincs XCom – natív Python adatátadás

### Döntési fa – mikor Prefect?

```
Van meglévő Airflow infrastruktúra?
├── Igen → maradjunk Airflow-nál (migráció drága)
└── Nem → Kell erős ökoszisztéma / sok előre gyártott integráció?
           ├── Igen → Airflow (több operator, provider)
           └── Nem → Fontos a Python-natív élmény és gyors iteráció?
                      ├── Igen → Prefect ✓
                      └── Kell asset-alapú lineage és data quality?
                             └── Igen → Dagster (→ Demo 3)
```

### Összefoglalás
| | Airflow | Prefect | Dagster |
|---|---|---|---|
| Paradigma | DAG (statikus) | Flow (dinamikus) | Asset (deklaratív) |
| Tanulási görbe | Meredek | Lapos | Közepes |
| Adatátadás | XCom | Return value | IO Manager |
| Cache | Nincs built-in | `task_input_hash` | Auto (asset freshness) |
| Legjobb | Meglévő Airflow env | Új Python projekt | Data platform / lineage |